# 1. 概要 / 使い方

このNotebookは、特徴量Excel・ターゲットExcel・LightGBMパラメータExcelを使って、n期先予測（回帰/分類）を検証します。

- 学習期間: `date <= CUTOFF_DATE`
- 予測期間: `date > CUTOFF_DATE`
- 回帰: 変化率を予測し、符号でフラグ化（`>=0 -> 1`, `<0 -> -1`）
- 分類: 変化フラグ `{-2,-1,1,2}` を4クラス分類
- `HORIZON_N=0` の場合は、ターゲットが評価日ベースで既に焼成済み（`t` に対して実績も `t`）として扱います。

設定変更は **3. 設定** セルのみで行ってください。

# 2. 環境・依存関係

In [ ]:
import ast
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, mean_squared_error

import openpyxl  # noqa: F401

print('pandas   :', pd.__version__)
print('numpy    :', np.__version__)
print('lightgbm :', lgb.__version__)
print('sklearn  :', sklearn.__version__)

# 3. 設定（ユーザが触る場所）

In [ ]:
# Notebook実行位置に依存しないように、プロジェクトルート候補を解決
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent

FEATURES_XLSX_PATH = PROJECT_ROOT / 'data' / 'features.xlsx'
TARGET_RETURN_XLSX_PATH = PROJECT_ROOT / 'data' / 'target_return.xlsx'
TARGET_FLAG_XLSX_PATH = PROJECT_ROOT / 'data' / 'target_flag.xlsx'
LGB_PARAMS_XLSX_PATH = PROJECT_ROOT / 'data' / 'lgb_params.xlsx'

DATE_COL_NAME = None  # Noneなら先頭列を日付列として使用
CUTOFF_DATE = '2024-12-31'
HORIZON_N = 5
TASK_MODE = 'regression_return'  # 'regression_return' or 'classification_flag'
RANDOM_SEED = 42

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TASK_MODE   :', TASK_MODE)
print('CUTOFF_DATE :', CUTOFF_DATE)
print('HORIZON_N   :', HORIZON_N)

# 4. 入出力ユーティリティ関数

In [ ]:
def _resolve_date_col(df: pd.DataFrame, date_col_name: str | None) -> str:
    if df.shape[1] < 2:
        raise ValueError('列数不足: 少なくとも2列（日付列 + 値列）が必要です。')
    if date_col_name is None:
        return df.columns[0]
    if date_col_name not in df.columns:
        raise ValueError(f'日付列 {date_col_name!r} が見つかりません。columns={list(df.columns)}')
    return date_col_name


def _prepare_datetime_index(df: pd.DataFrame, date_col_name: str | None) -> pd.DataFrame:
    date_col = _resolve_date_col(df, date_col_name)
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors='raise')
    out = out.sort_values(date_col).set_index(date_col)
    if out.index.has_duplicates:
        dupes = out.index[out.index.duplicated()].unique()
        sample = ', '.join(map(str, dupes[:5]))
        raise ValueError(f'重複日付を検出したため停止します。例: {sample}')
    return out


def load_features_excel(path: str | Path, date_col_name: str | None = None) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'特徴量ファイルが見つかりません: {p}')
    raw = pd.read_excel(p, engine='openpyxl')
    prepared = _prepare_datetime_index(raw, date_col_name)
    if prepared.shape[1] < 1:
        raise ValueError('特徴量列が存在しません。')
    return prepared


def load_target_excel(path: str | Path, date_col_name: str | None = None) -> pd.Series:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'ターゲットファイルが見つかりません: {p}')
    raw = pd.read_excel(p, engine='openpyxl')
    prepared = _prepare_datetime_index(raw, date_col_name)
    if prepared.shape[1] < 1:
        raise ValueError('ターゲット列が存在しません。')
    target_col = prepared.columns[0]
    y = prepared[target_col].copy()
    y.name = str(target_col)
    return y


def _parse_param_value(value: Any) -> Any:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == '':
        return None
    try:
        return ast.literal_eval(text)
    except Exception:
        return value


def load_lgb_params_excel(path: str | Path) -> dict[str, Any]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'LightGBMパラメータファイルが見つかりません: {p}')

    raw = pd.read_excel(p, engine='openpyxl')
    if raw.shape[1] < 2:
        raise ValueError('パラメータExcelは2列（parameter, value）以上が必要です。')

    key_col = raw.columns[0]
    val_col = raw.columns[1]

    params: dict[str, Any] = {}
    for _, row in raw.iterrows():
        key_raw = row[key_col]
        if pd.isna(key_raw):
            continue
        key = str(key_raw).strip()
        if key == '':
            continue

        parsed = _parse_param_value(row[val_col])
        if parsed is None:
            continue
        params[key] = parsed

    return params

# 5. データ読み込みと整合性チェック

In [ ]:
if TASK_MODE not in {'regression_return', 'classification_flag'}:
    raise ValueError('TASK_MODE は regression_return または classification_flag を指定してください。')
if int(HORIZON_N) < 0:
    raise ValueError('HORIZON_N は 0 以上の整数を指定してください。')

cutoff_ts = pd.to_datetime(CUTOFF_DATE)

features_df = load_features_excel(FEATURES_XLSX_PATH, date_col_name=DATE_COL_NAME)
target_path = TARGET_RETURN_XLSX_PATH if TASK_MODE == 'regression_return' else TARGET_FLAG_XLSX_PATH
target_s = load_target_excel(target_path, date_col_name=DATE_COL_NAME)
lgb_params = load_lgb_params_excel(LGB_PARAMS_XLSX_PATH)

print('--- Features ---')
print('shape:', features_df.shape)
print('date range:', features_df.index.min(), '->', features_df.index.max())
print('missing ratio per column (head):')
display(features_df.isna().mean().sort_values(ascending=False).head(10))

print('--- Target ---')
print('length:', len(target_s))
print('name:', target_s.name)
print('date range:', target_s.index.min(), '->', target_s.index.max())
print('missing ratio:', target_s.isna().mean())

common_index = features_df.index.intersection(target_s.index)
print('intersection length:', len(common_index))
if len(common_index) == 0:
    raise ValueError('featuresとtargetの日付交差が0件です。')

X_common = features_df.loc[common_index].sort_index()
y_common = target_s.loc[common_index].sort_index()

print('inner join後のshape:', X_common.shape, y_common.shape)
print('params loaded:', lgb_params)

# 6. n期先予測用のアラインメント

In [ ]:
def build_aligned_dataset(X: pd.DataFrame, y: pd.Series, horizon_n: int) -> tuple[pd.DataFrame, pd.Series]:
    if horizon_n < 0:
        raise ValueError('horizon_n は0以上である必要があります。')

    y_shifted = y.shift(-horizon_n)
    y_shifted.name = y.name if y.name is not None else 'target'

    joined = X.join(y_shifted.rename('__target__'), how='inner')
    X_aligned = joined[X.columns]
    y_aligned = joined['__target__']
    y_aligned.name = y_shifted.name
    return X_aligned, y_aligned


X_aligned, y_aligned = build_aligned_dataset(X_common, y_common, int(HORIZON_N))

train_mask = (X_aligned.index <= cutoff_ts) & y_aligned.notna()
test_mask_all = X_aligned.index > cutoff_ts
eval_mask = test_mask_all & y_aligned.notna()

X_train = X_aligned.loc[train_mask]
y_train = y_aligned.loc[train_mask]
X_test_all = X_aligned.loc[test_mask_all]
y_test_all = y_aligned.loc[test_mask_all]

print('aligned total          :', len(X_aligned))
print('train rows (y notna)   :', len(X_train))
print('test rows (all)        :', len(X_test_all))
print('eval rows (actual only):', int(eval_mask.sum()))

if len(X_train) == 0:
    raise ValueError('学習データが0件です。CUTOFF_DATE/HORIZON_N/入力データを確認してください。')
if len(X_test_all) == 0:
    warnings.warn('テスト期間 (date > cutoff_date) が0件です。')

# 7. モデル学習（LightGBM）

In [ ]:
params = dict(lgb_params)
params['random_state'] = int(RANDOM_SEED)

class_mapping = {-2: 0, -1: 1, 1: 2, 2: 3}
inverse_mapping = {v: k for k, v in class_mapping.items()}

if TASK_MODE == 'regression_return':
    if 'objective' not in params:
        params['objective'] = 'regression'

    for k in ['multiclass', 'num_class']:
        if k in params:
            warnings.warn(f'回帰モードのためパラメータ {k!r} は無視します。')
            params.pop(k, None)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train.astype(float))

else:
    if 'objective' not in params:
        params['objective'] = 'multiclass'
    if 'num_class' not in params:
        params['num_class'] = 4

    unique_train_labels = set(pd.Series(y_train).dropna().astype(int).unique().tolist())
    allowed = set(class_mapping.keys())
    if not unique_train_labels.issubset(allowed):
        raise ValueError(f'分類ターゲットに想定外ラベルがあります: {sorted(unique_train_labels - allowed)}')

    missing_classes = sorted(list(allowed - unique_train_labels))
    if missing_classes:
        warnings.warn(f'学習データに欠落クラスがあります: {missing_classes}（警告のみで継続）')

    y_train_encoded = y_train.astype(int).map(class_mapping)
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train_encoded)

print(model)

# 8. 予測・評価（混同行列）

In [ ]:
if len(X_test_all) == 0:
    raise ValueError('予測対象が0件のため評価できません。')

eval_index = y_test_all.dropna().index

if TASK_MODE == 'regression_return':
    y_pred_raw_all = pd.Series(model.predict(X_test_all), index=X_test_all.index, name='y_pred_raw')
    y_pred_flag_all = pd.Series(np.where(y_pred_raw_all >= 0, 1, -1), index=X_test_all.index, name='y_pred_flag')

    y_true_eval = y_test_all.loc[eval_index].astype(float)
    pred_flag_eval = y_pred_flag_all.loc[eval_index].astype(int)
    actual_flag_eval = pd.Series(np.where(y_true_eval >= 0, 1, -1), index=eval_index)

    cm_labels = [-1, 1]
    cm = confusion_matrix(actual_flag_eval, pred_flag_eval, labels=cm_labels)

    acc = accuracy_score(actual_flag_eval, pred_flag_eval) if len(eval_index) > 0 else np.nan
    f1m = f1_score(actual_flag_eval, pred_flag_eval, average='macro', labels=cm_labels, zero_division=0) if len(eval_index) > 0 else np.nan
    rmse = np.sqrt(mean_squared_error(y_true_eval, y_pred_raw_all.loc[eval_index])) if len(eval_index) > 0 else np.nan

    print('Accuracy :', acc)
    print('Macro F1 :', f1m)
    print('RMSE     :', rmse)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix (Regression Sign)')
    plt.show()

else:
    pred_class_all = pd.Series(model.predict(X_test_all), index=X_test_all.index, name='y_pred_raw').astype(int)
    y_pred_raw_all = pred_class_all
    y_pred_flag_all = pred_class_all.map(inverse_mapping).rename('y_pred_flag')

    y_true_eval = y_test_all.loc[eval_index].astype(int)
    pred_flag_eval = y_pred_flag_all.loc[eval_index].astype(int)

    cm_labels = [-2, -1, 1, 2]
    cm = confusion_matrix(y_true_eval, pred_flag_eval, labels=cm_labels)

    acc = accuracy_score(y_true_eval, pred_flag_eval) if len(eval_index) > 0 else np.nan
    f1m = f1_score(y_true_eval, pred_flag_eval, average='macro', labels=cm_labels, zero_division=0) if len(eval_index) > 0 else np.nan

    print('Accuracy :', acc)
    print('Macro F1 :', f1m)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix (Classification Flag)')
    plt.show()

print('Confusion matrix:')
print(cm)

# 9. 結果テーブル出力

In [ ]:
results_df = pd.DataFrame({
    'date': X_test_all.index,
    'y_true': y_test_all.values,
    'y_pred_raw': y_pred_raw_all.values,
    'y_pred_flag': y_pred_flag_all.values,
}).reset_index(drop=True)

cutoff_text = pd.to_datetime(CUTOFF_DATE).strftime('%Y%m%d')
out_name = f'pred_{TASK_MODE}_h{int(HORIZON_N)}_cutoff{cutoff_text}.csv'
out_path = OUTPUT_DIR / out_name
results_df.to_csv(out_path, index=False)

print('saved:', out_path)
display(results_df.head(10))

# 10. 再現性・注意点

- 欠損は埋めず、ターゲット欠損行のみ学習から除外します。
- 予測対象は `date > cutoff_date` です。
- 実績が無い行（`t + horizon_n` がデータ範囲外）は評価対象外ですが、予測結果テーブルには含まれます。
- `horizon_n=0` の場合、ターゲットは評価日 `t` と同日対応（シフトなし）として扱います。
- `horizon_n` を大きくすると、学習に使える実質的な最終日は `cutoff_date - horizon_n` 側に寄ります。